# 33 — Viền đen: đo thật, rồi chấm lại baseline

`32` để lại ba câu chưa tách được. Notebook này trả lời bằng **CPU, không GPU, không trích lại
embedding** — chỉ đọc shard letterbox và các file dự đoán `p_*.npy` đã lưu.

| câu hỏi | cách trả lời |
|---|---|
| viền đen chiếm bao nhiêu, hình dạng ra sao | quét bbox nội dung trên **toàn bộ val** (C1–C2) |
| resize không-đều + gom batch còn lại bao nhiêu đệm | dựng phân bố AR rồi mô phỏng ba cách (C3) |
| **đệm hay độ phân giải mới là thủ phạm** | phân tầng val theo tỉ lệ đệm, chấm `lb518` vs `crop518` trong từng tầng (C5) |
| kết quả `32` có đúng không | cài đặt lại phép chấm + gộp study **độc lập**, đối chiếu từng số (C4) |
| `cls` vs `patch` theo từng nhãn | C6 |
| EHR thêm được gì **cho từng nhãn** | C7 |

**Cách đọc C5 — đây là cell quyết định:**

- phạt **tập trung ở tầng nhiều đệm** → đệm là thủ phạm → sửa bằng resize không-đều, **không cần chạy `lb1022`**
- phạt **phẳng đều qua các tầng** → đệm không phải → độ phân giải là giả thuyết còn lại → chạy `lb1022`

In [1]:
import os, io, json, time, tarfile
import numpy as np, pandas as pd
from PIL import Image

try:
    from google.colab import drive
    drive.mount('/content/drive'); DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = './planB_out'

OUT_DIR = os.path.join(DATA_DIR, 'cxrlt_2024')
EMB_DIR = os.path.join(OUT_DIR, 'embeddings')
RES_DIR = os.path.join(OUT_DIR, 'stage3_baselines')
LB_DIR  = os.path.join(DATA_DIR, 'cxr_1022_letterbox_cxrlt')
PAD_DIR = os.path.join(OUT_DIR, 'stage3_padding'); os.makedirs(PAD_DIR, exist_ok=True)

W_FRONTAL, W_LATERAL = 0.7, 0.3      # phai KHOP 32; neu doi thi C4 se bat lech
LB_SIZE, PATCH = 1022, 14
BATCH_SIM = 64                       # batch gia dinh khi mo phong gom nhom o C3

# mount Drive co the "song" tren os.path.exists nhung da chet -> doc that mot file
assert os.path.isdir(LB_DIR), f'khong thay {LB_DIR} -> chay 28c truoc'
_t = sorted(f for f in os.listdir(LB_DIR) if f.endswith('.tar'))
assert _t, f'{LB_DIR} rong'
with open(os.path.join(LB_DIR, _t[0]), 'rb') as _f: assert len(_f.read(4096)) == 4096
print(f'shard letterbox: {len(_t)} tar (doc that OK)')

INV   = json.load(open(os.path.join(OUT_DIR, 'cxrlt_inventory.json')))
LAB40 = INV['labels_task1']
TIER_OF = {l: t for t in ('head', 'medium', 'tail') for l in INV['tier'][t]}

ORDER = pd.read_parquet(os.path.join(EMB_DIR, 'emb_index.parquet'))
Y = (pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_index.parquet'),
                     columns=['dicom_id'] + LAB40)
       .set_index('dicom_id').loc[ORDER.dicom_id])
assert (Y.index.to_numpy() == ORDER.dicom_id.to_numpy()).all()
YM = Y[LAB40].to_numpy('float32')

P_VIEW = os.path.join(OUT_DIR, 'cxrlt_view_resolved.parquet')
if os.path.exists(P_VIEW):
    V = (pd.read_parquet(P_VIEW)[['dicom_id', 'view_group', 'use']]
           .set_index('dicom_id').loc[ORDER.dicom_id].reset_index())
else:
    V = ORDER[['dicom_id', 'view_group']].copy(); V['use'] = V.view_group != 'unknown'

VA  = (ORDER.split == 'val').to_numpy()
DID_VA = ORDER.dicom_id.to_numpy()[VA]
YVA = YM[VA]
N_POS = {l: int(YVA[:, j].sum()) for j, l in enumerate(LAB40)}

# du doan da luu boi 32 — day la thu nb33 cham lai, KHONG fit lai gi ca
PRED_F = {f[2:-4]: os.path.join(RES_DIR, f)
          for f in sorted(os.listdir(RES_DIR)) if f.startswith('p_') and f.endswith('.npy')}
print(f'\n{len(ORDER):,} anh | val {VA.sum():,} anh')
print(f'du doan da luu ({len(PRED_F)}): {sorted(PRED_F)}')
assert PRED_F, 'chua co p_*.npy -> chay 32 truoc'

Mounted at /content/drive
shard letterbox: 378 tar (doc that OK)

377,110 anh | val 39,293 anh
du doan da luu (12): ['crop518_both', 'crop518_cls', 'crop518_patch', 'fusion_CXR-only', 'fusion_EHR-only', 'fusion_EHR_CXR', 'lb518_both', 'lb518_cls', 'lb518_patch', 'nosq518_both', 'nosq518_cls', 'nosq518_patch']


## C1 — Quét viền đen trên toàn bộ val

Đọc shard letterbox 1022, chỉ giải mã ảnh thuộc **val**. Dùng `Image.draft` để JPEG giải mã ở
**1/8 độ phân giải** (1022 → 128px): đủ để đo bề dày dải đen tới ~8px, mà nhanh hơn ~15 lần.

Đo **viền liên tục từ mép vào**, không phải tổng số hàng tối — đúng cách letterbox tạo ra dải, và
nó cũng gộp luôn phần collimation dính mép (phần đó cũng là diện tích lãng phí thật).

Đo ở **ba ngưỡng** (2 / 8 / 20) trong cùng một lượt để C2 kiểm xem kết quả có nhạy với ngưỡng không —
JPEG có ringing nên dải đen không về đúng 0.

Kết quả ghi cache; chạy lại là tức thì.

In [2]:
# ============ C1-THAY: doc song song + doc BULK + checkpoint tung tar ============
import concurrent.futures as cf, random
try:    from tqdm.auto import tqdm
except Exception: tqdm = lambda x, **k: x

CACHE = os.path.join(PAD_DIR, 'pad_val.parquet')
N_THREAD, RETRY = 8, 4          # Drive siet them thi HA xuong 4; 8 x 130MB ~ 1 GB RAM
THRS = (2, 8, 20)
PARTS = os.path.join(PAD_DIR, 'pad_parts'); os.makedirs(PARTS, exist_ok=True)
want  = set(DID_VA.tolist())
COLS  = ['dicom_id','H','W'] + [f'{k}{t}' for t in THRS for k in ('h','w','sym')]

def _bord(v, thr):
    m = v > thr
    if not m.any(): return len(v), len(v)
    return int(m.argmax()), int(m[::-1].argmax())

def _scan_tar(tp):
    out = os.path.join(PARTS, os.path.basename(tp).replace('.tar', '.parquet'))
    if os.path.exists(out):                      # da xong tu lan chay truoc
        return len(pd.read_parquet(out)), True
    last = None
    for k in range(RETRY):
        try:
            with open(tp, 'rb') as f: buf = f.read()      # MOT lan tuan tu, het nhay vun vat
            rows = []
            with tarfile.open(fileobj=io.BytesIO(buf)) as tar:
                for m in tar:
                    if not m.isfile(): continue
                    did = m.name.rsplit('/', 1)[-1][:-4]
                    if did not in want: continue
                    im = Image.open(io.BytesIO(tar.extractfile(m).read()))
                    im.draft('L', (128, 128))
                    a = np.asarray(im.convert('L')); H, W = a.shape
                    rmax, cmax = a.max(1), a.max(0)
                    rec = dict(dicom_id=did, H=H, W=W)
                    for th in THRS:
                        t, b = _bord(rmax, th); l, r = _bord(cmax, th)
                        rec[f'h{th}'] = (H - t - b) / H
                        rec[f'w{th}'] = (W - l - r) / W
                        rec[f'sym{th}'] = abs(t - b) / H + abs(l - r) / W
                    rows.append(rec)
            del buf
            pd.DataFrame(rows, columns=COLS).to_parquet(out, index=False)
            return len(rows), False
        except Exception as e:
            last = e; time.sleep(2 ** k + random.random())   # lui dan, khong dap lai Drive
    raise RuntimeError(f'{os.path.basename(tp)}: {type(last).__name__} {last}')

tars = sorted(os.path.join(LB_DIR, f) for f in os.listdir(LB_DIR) if f.endswith('.tar'))
tot, cached, bad, t0 = 0, 0, [], time.time()
with cf.ThreadPoolExecutor(N_THREAD) as ex:
    futs = {ex.submit(_scan_tar, tp): tp for tp in tars}
    bar = tqdm(cf.as_completed(futs), total=len(tars), desc='quet dem', unit='tar')
    for fu in bar:
        try:
            n, c = fu.result(); tot += n; cached += c
        except Exception as e:
            bad.append(str(e)); continue
        bar.set_postfix(anh=f'{tot:,}', cache=cached, loi=len(bad),
                        tocdo=f'{tot/max(time.time()-t0,1e-9):.0f}/s')
if bad:
    print(f'\n{len(bad)} tar loi -> CHAY LAI CELL NAY, no se bo qua phan da xong:')
    for b in bad[:5]: print('  ', b)

PAD = pd.concat([pd.read_parquet(os.path.join(PARTS, f))
                 for f in sorted(os.listdir(PARTS))], ignore_index=True)
print(f'\n{len(PAD):,} anh trong {(time.time()-t0)/60:.1f} phut')
assert not bad, 'con tar loi -> chay lai cell truoc khi di tiep'
assert len(PAD) == len(DID_VA), f'thieu {len(DID_VA)-len(PAD):,} anh val'
PAD.to_parquet(CACHE, index=False)

TH = 20
PAD['content']  = PAD[f'h{TH}'] * PAD[f'w{TH}']
PAD['dem']      = 1 - PAD.content
PAD['portrait'] = PAD[f'h{TH}'] >= PAD[f'w{TH}']
PAD['AR'] = (np.maximum(PAD[f'h{TH}'], PAD[f'w{TH}'])
             / np.minimum(PAD[f'h{TH}'], PAD[f'w{TH}']))
print(f'phu val: {len(PAD):,}/{VA.sum():,} = {len(PAD)/VA.sum():.2%}')


quet dem:   0%|          | 0/378 [00:00<?, ?tar/s]


39,293 anh trong 0.3 phut
phu val: 39,293/39,293 = 100.00%


## C2 — Cổng hình học

Nếu ta đang đo **đúng dải letterbox** thì ba điều phải đúng:

1. **đối xứng** — hai dải trên/dưới (hoặc trái/phải) bằng nhau, vì letterbox chia đôi phần thừa
2. **chỉ một trục** — trục còn lại phải kín, không có dải
3. **khớp con số đã biết** — diện tích nội dung trung bình phải ≈ **83.2%** (suy ra từ `1/AR` ở §7
   của `REVIEW_CXRLT_STAGE0.md`, tính độc lập từ metadata chứ không từ ảnh)

Điều 3 là kiểm chứng chéo mạnh nhất: hai đường tính hoàn toàn khác nhau mà ra cùng số.

Nếu cổng 1 hoặc 2 trượt thì thứ ta đo **không phải** đệm letterbox mà là collimation — và toàn bộ
phần sau phải đọc lại theo nghĩa khác.

In [3]:
print('=== do nhay theo NGUONG ===')
for th in THRS:
    c = (PAD[f'h{th}'] * PAD[f'w{th}'])
    print(f'  nguong {th:>2}: noi dung trung binh {c.mean():.4f} | dem {1-c.mean():.4f}')

print('\n=== cong 1: DOI XUNG ===')
print(f'  |tren-duoi|+|trai-phai| trung binh {PAD[f"sym{TH}"].mean():.4f} '
      f'| p95 {PAD[f"sym{TH}"].quantile(.95):.4f}')
_sym_ok = PAD[f'sym{TH}'].median() < 0.02

print('\n=== cong 2: CHI MOT TRUC ===')
_kin = 1 - np.maximum(PAD[f'h{TH}'], PAD[f'w{TH}'])       # thieu hut o truc DAI
print(f'  truc dai bi mat trung binh {_kin.mean():.4f} | p95 {np.quantile(_kin,.95):.4f}')
_axis_ok = np.median(_kin) < 0.02

print('\n=== cong 3: KHOP 83.2% doc lap ===')
print(f'  do duoc {PAD.content.mean():.4f}  vs  du kien 0.832  '
      f'| lech {PAD.content.mean()-0.832:+.4f}')
_match_ok = abs(PAD.content.mean() - 0.832) < 0.05

print(f'\nhuong anh: portrait {PAD.portrait.mean():.2%} | landscape {1-PAD.portrait.mean():.2%}')
print('\nphan vi ti le DEM:')
print(PAD.dem.describe(percentiles=[.05,.25,.5,.75,.95]).to_string(
      float_format=lambda v: f'{v:.4f}'))

for nm, ok in (('doi xung', _sym_ok), ('mot truc', _axis_ok), ('khop 83.2%', _match_ok)):
    print(f'  [{"OK " if ok else "TRUOT"}] {nm}')
if not (_sym_ok and _axis_ok and _match_ok):
    print('\n  !! It nhat mot cong truot -> thu do KHONG phai dem letterbox thuan tuy.')
    print('     Doc C5 nhu "phan tang theo ti le DEN" chu khong phai "theo dem letterbox".')

=== do nhay theo NGUONG ===
  nguong  2: noi dung trung binh 0.8138 | dem 0.1862
  nguong  8: noi dung trung binh 0.8128 | dem 0.1872
  nguong 20: noi dung trung binh 0.8112 | dem 0.1888

=== cong 1: DOI XUNG ===
  |tren-duoi|+|trai-phai| trung binh 0.0317 | p95 0.1484

=== cong 2: CHI MOT TRUC ===
  truc dai bi mat trung binh 0.0244 | p95 0.1484

=== cong 3: KHOP 83.2% doc lap ===
  do duoc 0.8112  vs  du kien 0.832  | lech -0.0208

huong anh: portrait 74.49% | landscape 25.51%

phan vi ti le DEM:
count   39293.0000
mean        0.1888
std         0.0809
min         0.0000
5%          0.0430
25%         0.1680
50%         0.1758
75%         0.2085
95%         0.3469
max         0.9550
  [OK ] doi xung
  [OK ] mot truc
  [OK ] khop 83.2%


## C3 — Còn lại bao nhiêu đệm nếu bỏ letterbox vuông

Ba cách, cạnh dài luôn cố định 1022, cạnh ngắn phải chia hết cho `patch = 14`:

| cách | canvas cạnh ngắn |
|---|---|
| **vuông** (hiện tại) | 1022 cho mọi ảnh |
| **K rổ theo phân vị AR** | mỗi rổ một canvas, đủ chứa ảnh *ít dẹt nhất* trong rổ |
| **sắp theo AR, batch liên tiếp** | mỗi batch một canvas |

Cách thứ ba không cần chọn `K`, không cần biết trước phân bố, và ảnh có tỉ lệ lạ tự nằm cạnh ảnh
gần nó nhất. Cột `patch/anh` cho biết luôn phần compute tiết kiệm được — attention theo `n²` nên
phần tiết kiệm ở đó còn lớn hơn.

Ảnh portrait và landscape **không xếp chung batch được** (canvas chuyển vị), nên gom nhóm luôn tách
theo hướng ảnh trước.

In [4]:
G = PAD.sort_values(['portrait', 'AR']).reset_index(drop=True)
short_px = LB_SIZE / G.AR.to_numpy()                    # canh ngan noi dung (px), canh dai = 1022

def _waste(canvas_short):
    canvas_short = np.ceil(canvas_short / PATCH) * PATCH
    w = 1 - short_px / canvas_short
    npatch = (LB_SIZE / PATCH) * (canvas_short / PATCH)
    return w, npatch

def _by_group(key):
    # canvas cua nhom = du chua anh IT DEP NHAT (AR nho nhat -> canh ngan lon nhat)
    hi = pd.Series(short_px).groupby(key).transform('max').to_numpy()
    return _waste(hi)

rows = []
w, np_ = _waste(np.full(len(G), float(LB_SIZE)))
rows.append(dict(cach='vuong 1022 (hien tai)', dem_tb=w.mean(), dem_p95=np.quantile(w,.95),
                 patch_anh=np_.mean()))
for K in (3, 5, 8, 16):
    key = (G.portrait.astype(int) * K
           + G.groupby('portrait').AR.transform(
               lambda s: np.clip(np.searchsorted(np.quantile(s, np.linspace(0,1,K+1)), s,
                                                 'right') - 1, 0, K-1)))
    w, np_ = _by_group(key.to_numpy())
    rows.append(dict(cach=f'{K} ro theo phan vi AR', dem_tb=w.mean(),
                     dem_p95=np.quantile(w,.95), patch_anh=np_.mean()))
key = G.portrait.astype(int).to_numpy() * 10**9 + np.arange(len(G)) // BATCH_SIM
w, np_ = _by_group(key)
rows.append(dict(cach=f'sap theo AR, batch {BATCH_SIM}', dem_tb=w.mean(),
                 dem_p95=np.quantile(w,.95), patch_anh=np_.mean()))

B = pd.DataFrame(rows)
B['giam_patch'] = B.patch_anh / B.patch_anh.iloc[0]
B['giam_attn']  = B.giam_patch ** 2
print('=== DEM CON LAI + COMPUTE, canh dai 1022, canh ngan boi so 14 ===')
print(B.to_string(index=False, float_format=lambda v: f'{v:.4f}'))
print(f'''
Doc: `dem_tb` la ti le dien tich canvas KHONG mang noi dung.
     `giam_attn` la ti le chi phi attention so voi letterbox vuong (theo n^2).
     San duoi cua moi cach la lam tron len boi so 14: {PATCH/LB_SIZE:.2%} moi hang patch.''')
B.to_csv(os.path.join(PAD_DIR, 'batching_design.csv'), index=False)

=== DEM CON LAI + COMPUTE, canh dai 1022, canh ngan boi so 14 ===
                 cach  dem_tb  dem_p95  patch_anh  giam_patch  giam_attn
vuong 1022 (hien tai)  0.1478   0.2305  5329.0000      1.0000     1.0000
 3 ro theo phan vi AR  0.0491   0.1406  4784.2895      0.8978     0.8060
 5 ro theo phan vi AR  0.0303   0.1013  4686.6137      0.8795     0.7734
 8 ro theo phan vi AR  0.0221   0.0640  4644.5653      0.8716     0.7596
16 ro theo phan vi AR  0.0159   0.0430  4613.9407      0.8658     0.7496
sap theo AR, batch 64  0.0077   0.0161  4576.0910      0.8587     0.7374

Doc: `dem_tb` la ti le dien tich canvas KHONG mang noi dung.
     `giam_attn` la ti le chi phi attention so voi letterbox vuong (theo n^2).
     San duoi cua moi cach la lam tron len boi so 14: 1.37% moi hang patch.


## C4 — Chấm lại toàn bộ baseline, cài đặt độc lập

Trước khi phân tầng, phải chắc phép chấm của `32` đúng. Cell này **cài đặt lại** hai thứ theo cách
khác hẳn:

- **AP** — tự viết bằng thứ hạng tích luỹ, không gọi `average_precision_score`
- **gộp study** — bằng `groupby` hai bước (trung bình trong từng view rồi trung bình có trọng số
  giữa các view), thay cho `bincount` của `32`

Rồi so từng con số với `cxr_only.csv`. Hai cài đặt độc lập ra cùng kết quả thì phép chấm là đúng;
lệch thì có lỗi ở một trong hai và phải tìm ra trước khi đọc bất cứ thứ gì khác.

In [5]:
def ap_rank(y, s):
    '''AP tu viet: sum (R_n - R_{n-1}) * P_n tren cac NGUONG PHAN BIET.
       Diem bang nhau phai gop lai thanh mot nguong — neu tach chung ra theo thu tu
       sap xep thi ket qua lech ~1e-7 so voi sklearn (da bat duoc o dry run).'''
    n1 = y.sum()
    if n1 == 0 or n1 == len(y): return np.nan
    o = np.argsort(-s, kind='stable'); yy = y[o]; ss = s[o]
    tp, fp = np.cumsum(yy), np.cumsum(1 - yy)
    last = np.r_[np.diff(ss) != 0, True]              # vi tri cuoi cua moi nhom diem bang nhau
    tp, fp = tp[last], fp[last]
    prec, rec = tp / (tp + fp), tp / n1
    return float((np.diff(np.r_[0.0, rec]) * prec).sum())

def mAP_img(p, mask=None, labs=None):
    labs = labs or LAB40
    y = YVA if mask is None else YVA[mask]
    q = p    if mask is None else p[mask]
    v = np.array([ap_rank(y[:, LAB40.index(l)], q[:, LAB40.index(l)]) for l in labs])
    return v

_S = pd.DataFrame(dict(sid=ORDER.study_id.to_numpy()[VA],
                       vg=V.view_group.to_numpy()[VA],
                       use=V.use.to_numpy()[VA]))
_WV = {'frontal': W_FRONTAL, 'lateral': W_LATERAL}

def pool_study_v2(p):
    '''Cai dat DOC LAP: trung binh trong (study,view) roi trung binh co trong so giua view.'''
    d = pd.DataFrame(p.astype('float64'), columns=LAB40)   # float32 cong don se lech ~1e-7
    d['sid'], d['vg'], d['use'] = _S.sid.to_numpy(), _S.vg.to_numpy(), _S.use.to_numpy()
    d = d[d.use & d.vg.isin(_WV)]
    g1 = d.groupby(['sid', 'vg'], sort=True)[LAB40].mean().reset_index()
    g1['w'] = g1.vg.map(_WV)
    num = g1[LAB40].mul(g1.w, axis=0).groupby(g1.sid).sum()
    den = g1.groupby('sid').w.sum()
    out = num.div(den, axis=0)
    ys = (pd.DataFrame(YVA, columns=LAB40).assign(sid=_S.sid.to_numpy())
            .groupby('sid')[LAB40].max().loc[out.index])
    return ys.to_numpy('float32'), out.to_numpy('float64')

CK = []
for nm, f in PRED_F.items():
    p = np.load(f)
    assert p.shape == (int(VA.sum()), len(LAB40)), (nm, p.shape)
    ai = mAP_img(p)
    ys, ps = pool_study_v2(p)
    asv = np.array([ap_rank(ys[:, j], ps[:, j]) for j in range(len(LAB40))])
    CK.append(dict(ten=nm, mAP_anh=np.nanmean(ai), mAP_study=np.nanmean(asv)))
CK = pd.DataFrame(CK)
print('=== CHAM LAI DOC LAP (tu p_*.npy da luu) ===')
print(CK.to_string(index=False, float_format=lambda v: f'{v:.4f}'))

P32 = os.path.join(RES_DIR, 'cxr_only.csv')
if os.path.exists(P32):
    R32 = pd.read_csv(P32).drop_duplicates(['arm', 'kind'], keep='last')
    R32['ten'] = R32.arm + '_' + R32.kind
    M = CK.merge(R32[['ten', 'mAP_anh', 'mAP_study']], on='ten', suffixes=('_33', '_32'))
    M['d_anh'] = M.mAP_anh_33 - M.mAP_anh_32
    M['d_study'] = M.mAP_study_33 - M.mAP_study_32
    print('\n=== DOI CHIEU VOI 32 ===')
    print(M[['ten','mAP_anh_33','mAP_anh_32','d_anh','mAP_study_33','mAP_study_32','d_study']]
          .to_string(index=False, float_format=lambda v: f'{v:+.6f}'))
    _e = max(M.d_anh.abs().max(), M.d_study.abs().max())
    print(f'\nlech lon nhat {_e:.2e}')
    if _e < 1e-6:  print('  [OK] hai cai dat doc lap khop -> phep cham cua 32 dung')
    elif _e < 5e-3: print('  [LUU Y] lech nho, kha nang do xu ly hoa/lam tron — doc duoc nhung nen tim ra')
    else:          print('  [TRUOT] lech that -> phai tim loi TRUOC khi doc C5')

=== CHAM LAI DOC LAP (tu p_*.npy da luu) ===
            ten  mAP_anh  mAP_study
   crop518_both   0.1855     0.1975
    crop518_cls   0.1638     0.1750
  crop518_patch   0.1917     0.2023
fusion_CXR-only   0.2028     0.2146
fusion_EHR-only   0.1199     0.1222
 fusion_EHR_CXR   0.2068     0.2184
     lb518_both   0.1974     0.2123
      lb518_cls   0.1656     0.1769
    lb518_patch   0.2002     0.2122
   nosq518_both   0.1974     0.2113
    nosq518_cls   0.1651     0.1760
  nosq518_patch   0.2028     0.2146

=== DOI CHIEU VOI 32 ===
          ten  mAP_anh_33  mAP_anh_32     d_anh  mAP_study_33  mAP_study_32   d_study
 crop518_both   +0.185455   +0.185455 -0.000000     +0.197529     +0.197529 -0.000000
  crop518_cls   +0.163817   +0.163817 -0.000000     +0.174970     +0.174970 +0.000000
crop518_patch   +0.191691   +0.191691 -0.000000     +0.202275     +0.202275 +0.000000
   lb518_both   +0.197444   +0.197444 -0.000000     +0.212314     +0.212314 -0.000000
    lb518_cls   +0.165623   +0.

## C5 — Cell quyết định: phân tầng val theo tỉ lệ đệm

Chia val thành **4 tầng theo tứ phân vị tỉ lệ đệm**, rồi chấm `lb518` và `crop518` **trong từng
tầng**. `crop518` không có đệm nào cả — nó là mốc so trong chính tầng đó, nên đã khử hết ảnh hưởng
của việc các tầng có prevalence khác nhau.

Chạy cho **cả `cls` lẫn `patch`**, vì đó mới là phép thử sắc: đệm được cho là phá `cls` (token gộp
toàn cục, −0.0254) nặng hơn hẳn `patch` (chỉ pha loãng, +0.0085). Nếu đúng, **độ dốc theo tầng của
`cls` phải dựng đứng hơn `patch`.**

`mAP` tuyệt đối giữa các tầng **không so được** với nhau (prevalence khác nhau) — chỉ cột `delta`
trong cùng một tầng mới có nghĩa.

In [6]:
_pos = {d: i for i, d in enumerate(DID_VA)}
_row = PAD.dicom_id.map(_pos)
_ok  = _row.notna()
ridx = _row[_ok].to_numpy(int)
pval = PAD.dem.to_numpy()[_ok.to_numpy()]
print(f'phu {len(ridx):,}/{len(DID_VA):,} anh val = {len(ridx)/len(DID_VA):.2%}')

QN = 4
qs = np.quantile(pval, np.linspace(0, 1, QN + 1)); qs[-1] += 1e-9
band = np.clip(np.searchsorted(qs, pval, 'right') - 1, 0, QN - 1)

def _strat(kind):
    a = np.load(PRED_F[f'crop518_{kind}']); b = np.load(PRED_F[f'lb518_{kind}'])
    out = []
    for q in range(QN):
        sel = ridx[band == q]
        m = np.zeros(len(DID_VA), bool); m[sel] = True
        va_ = mAP_img(a, m); vb_ = mAP_img(b, m)
        keep = ~(np.isnan(va_) | np.isnan(vb_))
        out.append(dict(tang=q + 1, dem_tb=pval[band == q].mean(), n_anh=int(m.sum()),
                        n_nhan=int(keep.sum()),
                        crop518=np.nanmean(va_[keep]), lb518=np.nanmean(vb_[keep]),
                        delta=np.nanmean(vb_[keep] - va_[keep])))
    return pd.DataFrame(out)

# San nhieu cua phep phan tang do bang HOAN VI, khong dung nguong doan. Dry run cho thay
# suy giam HOAN TOAN DEU van ra do doc +0.0117, tuc nguong cung 0.005 se bao dong gia.
NPERM = 20
def _slope(a, b, sel0, sel1):
    out = []
    for sel in (sel0, sel1):
        m = np.zeros(len(DID_VA), bool); m[sel] = True
        va_, vb_ = mAP_img(a, m), mAP_img(b, m)
        k = ~(np.isnan(va_) | np.isnan(vb_))
        out.append(np.nanmean(vb_[k] - va_[k]))
    return out[1] - out[0]

ST, DOC, SD = {}, {}, {}
for kind in ('cls', 'patch'):
    if f'crop518_{kind}' not in PRED_F or f'lb518_{kind}' not in PRED_F: continue
    T = _strat(kind); ST[kind] = T
    print(f'\n=== bieu dien `{kind}`: lb518 - crop518 theo TANG DEM ===')
    print(T.to_string(index=False, float_format=lambda v: f'{v:.4f}'))
    DOC[kind] = T.delta.to_numpy()[-1] - T.delta.to_numpy()[0]

    a = np.load(PRED_F[f'crop518_{kind}']); b = np.load(PRED_F[f'lb518_{kind}'])
    rs = np.random.default_rng(0); null = []
    for _ in range(NPERM):
        bb = rs.permutation(band)
        null.append(_slope(a, b, ridx[bb == 0], ridx[bb == QN - 1]))
    SD[kind] = float(np.std(null, ddof=1))
    print(f'  do doc (tang4 - tang1): {DOC[kind]:+.4f}')
    print(f'  san nhieu hoan vi ({NPERM} lan): sd {SD[kind]:.4f} | '
          f'z = {DOC[kind]/(SD[kind]+1e-12):+.2f}')

if ST:
    S2 = pd.concat([t.assign(kind=k, do_doc=DOC[k], sd_hoanvi=SD[k])
                    for k, t in ST.items()])
    S2.to_csv(os.path.join(PAD_DIR, 'stratified_padding.csv'), index=False)
    print('\n=== PHAN XU ===')
    for k, t in ST.items():
        print(f'  {k:<5} delta theo tang: ' + ' '.join(f'{v:+.4f}' for v in t.delta)
              + f'   | do doc {DOC[k]:+.4f} | z {DOC[k]/(SD[k]+1e-12):+.2f}')
    print('''
  DEM la thu pham  neu: do doc AM va z <= -2 (vuot han san nhieu hoan vi),
                        thuong kem theo `cls` doc hon `patch`.
                        -> sua bang resize khong-deu, KHONG can chay lb1022.
  DO PHAN GIAI     neu: |z| < 2 o ca hai bieu dien -> dem khong giai thich duoc.
                        -> chay lb1022 (2.3-3h GPU) moi tra loi duoc.
  z do tren san nhieu DO DUOC bang hoan vi nhan tang, khong phai nguong doan.''')

phu 39,293/39,293 anh val = 100.00%

=== bieu dien `cls`: lb518 - crop518 theo TANG DEM ===
 tang  dem_tb  n_anh  n_nhan  crop518  lb518  delta
    1  0.0919   8178      40   0.1645 0.1657 0.0012
    2  0.1688  10703      40   0.1674 0.1692 0.0018
    3  0.1884  10573      40   0.1602 0.1603 0.0001
    4  0.2916   9839      40   0.1623 0.1646 0.0023
  do doc (tang4 - tang1): +0.0011
  san nhieu hoan vi (20 lan): sd 0.0012 | z = +0.88

=== bieu dien `patch`: lb518 - crop518 theo TANG DEM ===
 tang  dem_tb  n_anh  n_nhan  crop518  lb518  delta
    1  0.0919   8178      40   0.1935 0.1999 0.0065
    2  0.1688  10703      40   0.1965 0.2056 0.0091
    3  0.1884  10573      40   0.1882 0.1971 0.0089
    4  0.2916   9839      40   0.1934 0.2049 0.0115
  do doc (tang4 - tang1): +0.0050
  san nhieu hoan vi (20 lan): sd 0.0035 | z = +1.44

=== PHAN XU ===
  cls   delta theo tang: +0.0012 +0.0018 +0.0001 +0.0023   | do doc +0.0011 | z +0.88
  patch delta theo tang: +0.0065 +0.0091 +0.0089 +0.011

## C6 — `cls` vs `patch` theo từng nhãn

Điều đã quan sát ở `32`: dùng `patch` thì một số medium và **toàn bộ tail** khá lên, còn head sụt.
Nhưng `32` chỉ in bảng theo nhãn cho trục **FOV** (crop vs letterbox), chưa bao giờ in cho trục
**biểu diễn**. Cell này in nó, để đọc từ dữ liệu chứ không từ trí nhớ.

In [7]:
R32 = pd.read_csv(os.path.join(RES_DIR, 'cxr_only.csv')).drop_duplicates(['arm','kind'], keep='last')
apc = [c for c in R32.columns if c.startswith('ap_')]
LBL = [c[3:] for c in apc]

for arm in sorted(R32.arm.unique()):
    sub = R32[R32.arm.eq(arm)]
    if not {'cls','patch'} <= set(sub.kind): continue
    c = sub[sub.kind.eq('cls')][apc].iloc[0].to_numpy('float64')
    p = sub[sub.kind.eq('patch')][apc].iloc[0].to_numpy('float64')
    T = pd.DataFrame(dict(label=LBL, cls=c, patch=p, delta=p - c))
    T['tier'] = T.label.map(TIER_OF); T['n_pos'] = T.label.map(N_POS)
    print(f'\n=== {arm}: patch - cls (mAP anh, tung nhan) ===')
    print(T.sort_values('delta', ascending=False)
           .to_string(index=False, float_format=lambda v: f'{v:.4f}'))
    print(f'\n--- {arm}: theo TIER ---')
    print(T.groupby('tier').agg(n=('delta','size'), delta_tb=('delta','mean'),
                                n_patch_thang=('delta', lambda s: int((s>0).sum())))
           .reindex(['head','medium','tail'])
           .to_string(float_format=lambda v: f'{v:+.4f}'))
    T.to_csv(os.path.join(PAD_DIR, f'cls_vs_patch_{arm}.csv'), index=False)
print('''
Doc: neu dau cua `delta` doi chieu theo TIER (tail duong, head am) thi viec chon bieu dien
     theo tung benh la co co so. Nhung chon tren val roi bao cao tren val la OVERFIT tieu chi
     chon — phai chon tren mot lat cat rieng tach tu train.''')


=== crop518: patch - cls (mAP anh, tung nhan) ===
                     label    cls  patch   delta   tier  n_pos
    Subcutaneous Emphysema 0.2856 0.5149  0.2293 medium    431
                    Hernia 0.4050 0.5320  0.1270 medium    674
              Pneumothorax 0.3765 0.4497  0.0732   head   2342
                      Mass 0.1127 0.1823  0.0696 medium    789
         Pneumomediastinum 0.0091 0.0632  0.0541   tail    122
                 Emphysema 0.1598 0.2110  0.0512 medium    741
                  Kyphosis 0.0283 0.0765  0.0481   tail    112
         Hydropneumothorax 0.0107 0.0555  0.0447   tail    128
          Pleural Effusion 0.7601 0.7899  0.0297   head  10430
Calcification of the Aorta 0.0438 0.0712  0.0274 medium    594
                    Nodule 0.0686 0.0951  0.0265 medium   1119
                  Fracture 0.0851 0.1110  0.0259   head   1576
                     Edema 0.4918 0.5175  0.0257   head   5946
                    Normal 0.2973 0.3224  0.0251   head   5088
    

## C7 — EHR thêm được gì, **cho từng nhãn**

`32` chỉ báo macro: `+0.0048`. Câu chưa trả lời là con số đó **trải mỏng đều 40 nhãn** hay **vài
nhãn được lợi thật còn lại bằng 0**. Hai kịch bản dẫn tới hai kết luận trái ngược, và nó quyết định
FiLM nên nhắm toàn bộ 40 nhãn hay chỉ một nhóm.

Nhắc lại giới hạn của phép đo: `32` nối vector rồi cho qua **một lớp tuyến tính**, tức
`W_e·e + W_h·h` — **không có số hạng chéo**. Nó không nói được gì về điều biến kiểu FiLM
(`γ(e) ⊙ h`), thứ mà lớp tuyến tính trên vector nối về mặt cấu trúc không biểu diễn được.

In [8]:
NEED = ['p_fusion_CXR-only.npy', 'p_fusion_EHR-only.npy', 'p_fusion_EHR_CXR.npy']
have = [n for n in NEED if os.path.exists(os.path.join(RES_DIR, n))]
if len(have) < 3:
    print(f'thieu {set(NEED)-set(have)} -> chay C6 cua 32 truoc')
else:
    pc = np.load(os.path.join(RES_DIR, 'p_fusion_CXR-only.npy'))
    pe = np.load(os.path.join(RES_DIR, 'p_fusion_EHR-only.npy'))
    pf = np.load(os.path.join(RES_DIR, 'p_fusion_EHR_CXR.npy'))
    ac, ae, af = mAP_img(pc), mAP_img(pe), mAP_img(pf)
    T = pd.DataFrame(dict(label=LAB40, cxr=ac, ehr=ae, fusion=af, lift=af - ac))
    T['tier'] = T.label.map(TIER_OF); T['n_pos'] = T.label.map(N_POS)
    print('=== lift cua EHR tren nen CXR, tung nhan ===')
    print(T.sort_values('lift', ascending=False)
           .to_string(index=False, float_format=lambda v: f'{v:.4f}'))
    print('\n--- theo TIER ---')
    print(T.groupby('tier').agg(n=('lift','size'), lift_tb=('lift','mean'),
                                n_duong=('lift', lambda s: int((s>0).sum())))
           .reindex(['head','medium','tail']).to_string(float_format=lambda v: f'{v:+.4f}'))
    NOISE = 0.005
    big = T[T.lift > NOISE].sort_values('lift', ascending=False)
    print(f'\nnhan co lift > nguong nhieu {NOISE}: {len(big)}/40'
          + (f' -> {", ".join(big.label)}' if len(big) else ''))
    print(f'macro lift {T.lift.mean():+.4f} | {int((T.lift>0).sum())}/40 nhan duong')
    print(f'''
Doc: tap trung o vai nhan co lien he sinh ly (BNP/troponin -> tim, WBC/vi sinh -> nhiem trung)
     -> tin hieu EHR THAT nhung bi macro trung binh hoa; FiLM nen nham dung nhom do.
     Trai mong deu ~{T.lift.mean():.4f} moi nhan -> du thua that o dang CONG TINH,
     nhung van chua noi duoc gi ve dang DIEU BIEN.''')
    T.to_csv(os.path.join(PAD_DIR, 'ehr_lift_per_label.csv'), index=False)

=== lift cua EHR tren nen CXR, tung nhan ===
                     label    cxr    ehr  fusion    lift   tier  n_pos
              Tuberculosis 0.0638 0.0875  0.0903  0.0266 medium    377
              Pneumothorax 0.4507 0.2146  0.4717  0.0210   head   2342
                 Pneumonia 0.2788 0.2524  0.2981  0.0193   head   7061
              Rib Fracture 0.0790 0.0656  0.0962  0.0172 medium   1250
               Lung Lesion 0.0346 0.0312  0.0489  0.0143 medium    314
                      Mass 0.1861 0.0588  0.2001  0.0140 medium    789
                  Fracture 0.1165 0.0712  0.1299  0.0134   head   1576
              Lung Opacity 0.5341 0.4171  0.5439  0.0098   head  11731
                     Edema 0.5180 0.3715  0.5244  0.0064   head   5946
              Infiltration 0.0562 0.0563  0.0622  0.0060   head   1506
             Consolidation 0.1770 0.1278  0.1824  0.0054   head   2379
Enlarged Cardiomediastinum 0.1599 0.1377  0.1645  0.0046   head   4244
        Pulmonary Embolism 0.010

## C8 — Tổng hợp: chạy gì tiếp

In [9]:
print('=' * 66)
print('TOM TAT 33')
print('=' * 66)
print(f'1. DEM DEN: trung binh {PAD.dem.mean():.2%} dien tich canvas | p95 {PAD.dem.quantile(.95):.2%}')
print(f'   phan vi: ' + ' '.join(f'{PAD.dem.quantile(q):.3f}' for q in (.05,.25,.5,.75,.95)))
print(f'\n2. BO DEM: ' + ' | '.join(
      f'{r.cach} -> {r.dem_tb:.2%}' for r in B.itertuples()))
if ST:
    print('\n3. PHAN TANG (cell quyet dinh):')
    _z = {k: DOC[k] / (SD[k] + 1e-12) for k in ST}
    for k in ST:
        verd = ('DEM la thu pham' if _z[k] <= -2 else
                'DEM khong phai — nghi DO PHAN GIAI' if abs(_z[k]) < 2 else
                'nguoc chieu du doan — doc lai')
        print(f'   {k:<5} do doc {DOC[k]:+.4f} (sd hoan vi {SD[k]:.4f}, z {_z[k]:+.2f})'
              f'  -> {verd}')
    if any(v <= -2 for v in _z.values()) and not any(v >= 2 for v in _z.values()):
        print('''
   => LAM: doi 28c/31 sang resize GIU TI LE + gom batch theo AR.
      KHONG can chay lb1022 truoc.''')
    elif all(abs(v) < 2 for v in _z.values()):
        print('''
   => LAM: dem khong giai thich duoc. Chay lb1022 (2.3-3h GPU) de kiem do phan giai.''')
    else:
        print('''
   => LAM: hai bieu dien khong cung ket luan — doc bang C5 truoc khi quyet.''')
print('''
Moi bang da ghi vao stage3_padding/:
  pad_val.parquet | batching_design.csv | stratified_padding.csv
  cls_vs_patch_*.csv | ehr_lift_per_label.csv''')

TOM TAT 33
1. DEM DEN: trung binh 18.88% dien tich canvas | p95 34.69%
   phan vi: 0.043 0.168 0.176 0.209 0.347

2. BO DEM: vuong 1022 (hien tai) -> 14.78% | 3 ro theo phan vi AR -> 4.91% | 5 ro theo phan vi AR -> 3.03% | 8 ro theo phan vi AR -> 2.21% | 16 ro theo phan vi AR -> 1.59% | sap theo AR, batch 64 -> 0.77%

3. PHAN TANG (cell quyet dinh):
   cls   do doc +0.0011 (sd hoan vi 0.0012, z +0.88)  -> DEM khong phai — nghi DO PHAN GIAI
   patch do doc +0.0050 (sd hoan vi 0.0035, z +1.44)  -> DEM khong phai — nghi DO PHAN GIAI

   => LAM: dem khong giai thich duoc. Chay lb1022 (2.3-3h GPU) de kiem do phan giai.

Moi bang da ghi vao stage3_padding/:
  pad_val.parquet | batching_design.csv | stratified_padding.csv
  cls_vs_patch_*.csv | ehr_lift_per_label.csv
